In [6]:
import subprocess

In [7]:
subprocess.run(["pip", "install", "pymupdf", "sentence-transformers", 
                "chromadb", "--quiet", "--break-system-packages"])

CompletedProcess(args=['pip', 'install', 'pymupdf', 'sentence-transformers', 'chromadb', '--quiet', '--break-system-packages'], returncode=0)

In [8]:
import fitz  # pymupdf
import chromadb
from sentence_transformers import SentenceTransformer
import re

print("Dependencies loaded")

/opt/miniconda3/envs/ml_month3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dependencies loaded


In [9]:
import urllib.request
import os

In [10]:
url = "https://www.apple.com/newsroom/pdfs/fy2023-q4/FY23_Q4_Consolidated_Financial_Statements.pdf"
filepath = "apple_q4_fy23.pdf"

In [11]:
if not os.path.exists(filepath):
    print("Downloading Apple Q4 FY23 financial statements...")
    urllib.request.urlretrieve(url, filepath)
    print(f"Downloaded: {filepath}")
else:
    print(f"Already exists: {filepath}")

Already exists: apple_q4_fy23.pdf


In [12]:
# Check file size
size_kb = os.path.getsize(filepath) / 1024
print(f"File size: {size_kb:.1f} KB")

File size: 3913.3 KB


# Extract THe TExt 

In [13]:
def extract_text_from_pdf(filepath: str) -> list[dict]:
    """
    we used list of dict  because we want to track which  page each chunk came from for citations like "this answer came form page 4 of this doc"
    """
    doc = fitz.open(filepath)
    pages = []

    for page_num  in range(len(doc)):
        page = doc[page_num]

        #Extract raw text 
        text = page.get_text()

        #Clean the text 
        text = re.sub(r'\n+', '\n' , text) # Multiple newlines -> single
        text = re.sub(r' +', ' ', text) # Multiple spaces into single space 
        text = text.strip()

        if len(text) > 50:
            pages.append({
                "page_num": page_num + 1, 
                "text": text,
                "char_count": len(text)
            })

    doc.close()
    return pages

In [14]:
pages = extract_text_from_pdf("apple_q4_fy23.pdf")

In [15]:
print(f"Total pages extracted: {len(pages)}")
print(f"Total characters: {sum(p['char_count'] for p in pages):,}")
print(f"\nFirst page preview:")
print(pages[0]['text'][:500])
print("...")
print(f"\nPage 1 char count: {pages[0]['char_count']}")

Total pages extracted: 3
Total characters: 6,071

First page preview:
Apple Inc. 
CONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS (Unaudited) 
(In millions, except number of shares, which are reflected in thousands, and per-share amounts) 
 
Three Months Ended 
 
Twelve Months Ended 
 
September 30, 
2023 
 
September 24, 
2022 
 
September 30, 
2023 
 
September 24, 
2022 
Net sales: 
 
 
 
 
 Products 
! 
67,184 ! 
70,958 ! 
298,085 ! 
316,199 
 Services 
 
22,314 
19,188 
85,200 
78,129 
Total net sales (1) 
 
89,498 
90,146 
383,285 
394,328 
Cost of sales: 
 
...

Page 1 char count: 2205


In [16]:
# Download a real 10-K from SEC EDGAR — text-based, clean extraction

In [17]:
url = "https://www.sec.gov/Archives/edgar/data/320193/000032019323000106/aapl-20230930.htm"
filepath = "apple_10k_2023.htm"

In [18]:
# this will not work because we sec blocks requests without user agent header 

# if not os.path.exists(filepath):
#     print("Downloading Apple 10-K 2023 from SEC EDGAR...")
#     urllib.request.urlretrieve(url, filepath)
#     print(f"Downloaded: {filepath}")

# size_mb = os.path.getsize(filepath) / (1024 * 1024)
# print(f"File size: {size_mb:.1f} MB")

In [19]:
import requests

In [20]:
if not os.path.exists(filepath):
    print("Downloading Apple 10-K 2023 from SEC EDGAR...")
    # WHY User-Agent: SEC EDGAR requires it by law
    # they block bots without identification
    headers = {
        "User-Agent": "Sonu Verma sonuverma@email.com",
        "Accept-Encoding": "gzip, deflate",
        "Host": "www.sec.gov"
    }
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        with open(filepath, 'wb') as f:
            f.write(response.content)
        print(f"Downloaded successfully")
    else:
        print(f"Failed: {response.status_code}")

size_mb = os.path.getsize(filepath) / (1024 * 1024)
print(f"File size: {size_mb:.1f} MB")

File size: 1.5 MB


# Extract Text from html 

In [21]:
from bs4 import BeautifulSoup

NameError: name 'apple_10k_2023' is not defined

In [91]:
class TableToNaturalLanguage:
    def __init__(self,company):
        """
        Initializes the converter with specific document metadata.
        """
        self.company = company
        
    def is_financial_table(self,table_soup):
        """
        Analyzes a table to determine if it contains financial data 
        based on keyword presence and numerical patterns.
        """
        financial_keywords = ["net", "revenue", "income", "sales", "earnings", 
                          "profit", "loss", "assets", "cash", "cost"]
        numeric_pattern =re.compile(r'\$?[\d,]+\.?\d*')

        rows = table_soup.find_all("tr")

        for row in rows:
            cells = row.find_all("td")
            if len(cells) < 2:
                continue

            first_cell_text = cells[0].get_text(strip=True).lower()
            has_keyword = any(kw in first_cell_text for kw in financial_keywords)

            has_numeric_value = False
            for cell in cells[1:]:
                cell_text = cell.get_text(strip=True)
                # We look for a cell that isn't empty and matches our financial number pattern
                if cell_text and numeric_pattern.search(cell_text):
                    has_numeric_value = True
                    break

            # If a single row contains both a keyword and a financial number, it's a winner
            if has_keyword and has_numeric_value:
                return True

        return False

    
    def convert(self, raw_html):
        soup = BeautifulSoup(raw_html, "html.parser")
        if not self.is_financial_table(soup):
            return []

        rows = soup.find_all("tr")
        col_map = {} # Temporary to find years and their order
        year_pattern = re.compile(r'20\d{2}')
        results = []

        # 1. Header Scan (Still needed to find WHICH years are in the table)
        for row in rows:
            cells = row.find_all(["td", "th"])
            for i, cell in enumerate(cells):
                text = cell.get_text(strip=True)
                if year_pattern.fullmatch(text):
                    col_map[i] = text
            if col_map: break

        if not col_map: return []
        
        # The One-Liner: Get years in the order they appear left-to-right
        years_in_order = [year for idx, year in sorted(col_map.items())]

        # 2. Data Extraction with Filtered Zip
        for row in rows:
            cells = row.find_all("td")
            if len(cells) < 2: continue
            
            metric = cells[0].get_text(strip=True)
            if not metric: continue

            # Extract only cells that look like real financial data (3+ digits)
            numeric_values = []
            for cell in cells[1:]:
                val_text = cell.get_text(strip=True)
                # Filtering out (4), 11, $, etc. while keeping 162,560
                if re.search(r'\d{3,}', val_text):
                    numeric_values.append(val_text)
            
            # 3. Zip 'em up! The order is now guaranteed.
            for value, year in zip(numeric_values, years_in_order):
                sentence = f"The {metric} of {self.company} in the fiscal year {year} was {value}."
                results.append(sentence)
                
        return results

In [85]:
def extract_text_from_html(filepath: str,table_converter) -> list[dict]:
    """
    beautiful soup parses html structure properly lets us target specific tags instead of raw text dump
    """

    with open(filepath,'r', encoding='utf-8' , errors='ignore') as f:
        html = f.read()

    soup = BeautifulSoup(html, 'html.parser')

    # Removing tags in html because it adds extra noise not content bb
    for tag in soup(['script', 'style', 'meta', 'link']):
        tag.decompose()

    # Every top level dv is a section html  doesnt have pages it have sections div and more
    sections = []

    # get all paragraph and table text
    elements = soup.find_all(['p','h1','h2','h3','table'])

    current_section = []
    current_length = 0
    section_num = 1

    for elem in elements:
        if elem.name == 'table':
            table_sentences = table_converter.convert(str(elem))
            if not table_sentences:
                continue
            text = " ".join(table_sentences) 
        else:
            text = elem.get_text(strip=True)

        if len(text) <  10:
            continue

        current_section.append(text)
        current_length += len(text)

        if current_length > 2000:
            sections.append({
                "section_num": section_num,
                "text": " ".join(current_section),
                "char_count": current_length
            })
            section_num += 1
            current_section = []
            current_length = 0

    if current_section:
        sections.append({
            "section_num":section_num,
            "text": " ".join(current_section),
            "char_count": current_length
        })
    return sections

In [92]:
# Simulated Americas data with inconsistent spacers
test_html = """
<table>
  <tr>
    <td></td>
    <td>2023</td>
    <td></td>
    <td></td>
    <td></td>
    <td>2022</td>
    <td></td>
    <td></td>
    <td></td>
    <td>2021</td>
  </tr>
  <tr>
    <td>Americas</td>
    <td>$</td>
    <td>162,560</td>
    <td></td>
    <td>$</td>
    <td>169,658</td>
    <td></td>
    <td></td>
    <td></td>
    <td>153,306</td>
  </tr>
</table>
"""

converter = TableToNaturalLanguage(company="Apple")
prose_results = converter.convert(test_html)

print("\n--- Processed Results ---")
for s in prose_results:
    print(s)


--- Processed Results ---


In [93]:
converter = TableToNaturalLanguage(
    company="Apple"
)

In [94]:
sections = extract_text_from_html("apple_10k_2023.htm",converter)

In [95]:
print(f"Total sections extracted: {len(sections)}")
print(f"Total characters: {sum(s['char_count'] for s in sections):,}")
print(f"\nFirst section preview:")
print("\n--- Checking for financial conversions ---")
for s in sections:
    if "fiscal year" in s["text"]:
        print(s["text"][:500])
        print("---")
        break

print("\n--- Checking California still appears ---")
california_found = any("California" in s["text"] for s in sections)
print(f"California in output: {california_found}")
print(sections[0]['text'][:500])

Total sections extracted: 10
Total characters: 25,206

First section preview:

--- Checking for financial conversions ---
The Americas of Apple in the fiscal year 2023 was 162,560. The Americas of Apple in the fiscal year 2022 was 169,658. The Americas of Apple in the fiscal year 2021 was 153,306. The Europe of Apple in the fiscal year 2023 was 94,294. The Europe of Apple in the fiscal year 2022 was 95,118. The Europe of Apple in the fiscal year 2021 was 89,307. The Greater China of Apple in the fiscal year 2023 was 72,559. The Greater China of Apple in the fiscal year 2022 was 74,200. The Greater China of Apple in
---

--- Checking California still appears ---
California in output: False
The Americas of Apple in the fiscal year 2023 was 162,560. The Americas of Apple in the fiscal year 2022 was 169,658. The Americas of Apple in the fiscal year 2021 was 153,306. The Europe of Apple in the fiscal year 2023 was 94,294. The Europe of Apple in the fiscal year 2022 was 95,118. The Europe of

In [96]:
# Inspect all sections to understand content quality
for i, section in enumerate(sections):
    print(f"Section {section['section_num']}: {section['char_count']} chars")
    print(f"  Preview: {section['text'][:150]}")
    print()

Section 1: 2257 chars
  Preview: The Americas of Apple in the fiscal year 2023 was 162,560. The Americas of Apple in the fiscal year 2022 was 169,658. The Americas of Apple in the fis

Section 2: 2076 chars
  Preview: The Research and development of Apple in the fiscal year 2023 was 29,915. The Research and development of Apple in the fiscal year 2022 was 26,251. Th

Section 3: 2922 chars
  Preview: The Net income of Apple in the fiscal year 2023 was 96,995. The Net income of Apple in the fiscal year 2022 was 99,803. The Net income of Apple in the

Section 4: 2696 chars
  Preview: The Land and buildings of Apple in the fiscal year 2023 was 23,446. The Land and buildings of Apple in the fiscal year 2022 was 22,126. The Machinery,

Section 5: 3434 chars
  Preview: The Current of Apple in the fiscal year 2023 was 9,445. The Current of Apple in the fiscal year 2022 was 7,890. The Current of Apple in the fiscal yea

Section 6: 2405 chars
  Preview: The Tax credit carryforwards of Apple in t

In [97]:
def chunk_sections(sections: list[dict], min_quality_score: float = 0.3) -> list[dict]:
    """
    garbage chunks waste embedding space and corrupt  retrieval results with irrelevant content
    """

    financial_signals = [
        "revenue", "income", "profit", "loss", "sales", "earnings",
        "expense", "cost", "margin", "cash", "debt", "equity",
        "assets", "liabilities", "operating", "net", "gross", "total"
    ]

    chunks = []

    for section in sections:
        text_lower = section["text"].lower()

        # QUAlity score = ratio of financial signal words present 
        signals_found = sum(1 for word in financial_signals if word in text_lower)
        quality_score = signals_found / len(financial_signals)

        # 0.3 threshold : at least 30% of signal words must appear filters  boilerplate while keeping financial content
        if quality_score >= min_quality_score:
            chunks.append({
                "chunk_id": f"chunk_{section['section_num']:03d}",
                "text": section["text"],
                "char_count": section["char_count"],
                "quality_score": round(quality_score,3),
                "source": "apple_10k_2023",
                "section_num": section["section_num"]
            })

    return chunks

In [98]:
chunks = chunk_sections(sections)

In [99]:
print(f"Sections before filtering: {len(sections)}")
print(f"Chunks after filtering: {len(chunks)}")
print(f"\nChunks kept:")
for chunk in chunks:
    print(f"  {chunk['chunk_id']}: quality={chunk['quality_score']} | {chunk['text'][:100]}")

Sections before filtering: 10
Chunks after filtering: 5

Chunks kept:
  chunk_002: quality=0.333 | The Research and development of Apple in the fiscal year 2023 was 29,915. The Research and developme
  chunk_003: quality=0.333 | The Net income of Apple in the fiscal year 2023 was 96,995. The Net income of Apple in the fiscal ye
  chunk_004: quality=0.389 | The Land and buildings of Apple in the fiscal year 2023 was 23,446. The Land and buildings of Apple 
  chunk_006: quality=0.389 | The Tax credit carryforwards of Apple in the fiscal year 2023 was 8,302. The Tax credit carryforward
  chunk_010: quality=0.389 | The Segment operating income of Apple in the fiscal year 2023 was 150,888. The Segment operating inc


# Load embedding model

In [100]:
# MiniLm because it us 6x faaster than mpnet 

In [101]:
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded")

Loading embedding model...


Loading weights: 100%|█████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 6273.70it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded


In [102]:
# Peristent clinet because  we want the db to survive differnt restarts
client = chromadb.PersistentClient(path="./chroma_db")

In [103]:
try:
    client.delete_collection("financial_docs")
    print("Deleted existing collection")
except:
    pass

Deleted existing collection


In [104]:
collection = client.create_collection(
    name="financial_docs",
    metadata={"hnsw:space": "cosine"}
)

In [105]:
print("Collection created")

Collection created


In [106]:
# Embed and store all chunks
print(f"\nEmbedding {len(chunks)} chunks...")
texts = [chunk["text"] for chunk in chunks]
embeddings = model.encode(texts, batch_size=32, show_progress_bar=True)


Embedding 5 chunks...


Batches: 100%|███████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.05it/s]


In [107]:
print(f"Embedding shape: {embeddings.shape}")

Embedding shape: (5, 384)


In [108]:
# Store embeddings in ChromaDB with metadata
collection.add(
    ids=[chunk["chunk_id"] for chunk in chunks],
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=[{
        "source": chunk["source"],
        "section_num": chunk["section_num"],
        "quality_score": chunk["quality_score"],
        "char_count": chunk["char_count"]
    } for chunk in chunks]
)
print(f"Stored {collection.count()} chunks in ChromaDB")

Stored 5 chunks in ChromaDB


In [111]:
print("\nTest retrieval — 'Apple revenue and net income'")
results = collection.query(
    query_texts=["Apple revenue and net income"],
    n_results=3
)

print(f"\nTop 3 results:")
for i, (doc, metadata, distance) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0], 
    results["distances"][0]
)):
    print(f"\nResult {i+1}:")
    print(f"  Section: {metadata['section_num']} | Quality: {metadata['quality_score']} | Distance: {round(distance, 4)}")
    print(f"  Preview: {doc[:200]}")


Test retrieval — 'Apple revenue and net income'

Top 3 results:

Result 1:
  Section: 10 | Quality: 0.389 | Distance: 0.3023
  Preview: The Segment operating income of Apple in the fiscal year 2023 was 150,888. The Segment operating income of Apple in the fiscal year 2022 was 152,895. The Segment operating income of Apple in the fisca

Result 2:
  Section: 2 | Quality: 0.333 | Distance: 0.3754
  Preview: The Research and development of Apple in the fiscal year 2023 was 29,915. The Research and development of Apple in the fiscal year 2022 was 26,251. The Research and development of Apple in the fiscal 

Result 3:
  Section: 3 | Quality: 0.333 | Distance: 0.4026
  Preview: The Net income of Apple in the fiscal year 2023 was 96,995. The Net income of Apple in the fiscal year 2022 was 99,803. The Net income of Apple in the fiscal year 2021 was 94,680. The Weighted-average


## RAG Ingestion Pipeline

### What I built
- PDF text extraction with PyMuPDF
- HTML extraction from SEC EDGAR with BeautifulSoup
- Quality filtering using financial signal words (0.3 threshold)
- Embedding with MiniLM (6 chunks × 384 dimensions)
- ChromaDB storage with metadata for citations

### Pipeline flow
PDF/HTML → Extract text → Clean → Section by 2000 chars → 
Quality filter → Embed → Store in ChromaDB

### Key decisions and why
- HTML over PDF for SEC filings: semantic structure preserved
- 2000 char sections: balance context vs chunk granularity
- Quality filter: removes boilerplate that corrupts retrieval
- Cosine distance: magnitude-independent similarity
- PersistentClient: survives restarts

### Problems found
- Dense embeddings alone insufficient: stock repurchase ranked #2 for revenue query
- Keyword quality scoring misses semantically similar boilerplate
- Fix: hybrid search (BM25 + dense) on Day 62

### Real numbers
- Apple 10-K: 1.5MB HTML, 13 sections, 6 chunks after filtering
- Embedding time: near instant for 6 chunks
- Top retrieval distance: 0.5567 (room for improvement)

In [112]:
print(collection.name)
print(collection.count())

financial_docs
5
